# TIDA Neuron Phase Locking Value (PLV) Analysis

This notebook implements a full pipeline to quantify **phase synchrony** between TIDA (Tuberoinfundibular Dopamine) neuron calcium imaging traces using the **Phase Locking Value (PLV)**.

## What is PLV?

The Phase Locking Value measures the consistency of the **phase difference** between two oscillatory signals over time:

- PLV = **1** → perfectly consistent phase relationship (strong synchrony)
- PLV = **0** → no consistent phase relationship

Note: PLV = 1 does **not** imply in-phase coupling — two antiphase cells (180° apart) also yield PLV = 1. Use the mean phase difference to characterise the nature of the coupling.


In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from scipy.stats import pearsonr

# Local analysis functions
from functions.utils import (
    bandpass,
    get_phase,
    compute_spectra,
    plv_einsum,
    permutation_test,
    circular_shift_surrogate,
    phase_randomisation_surrogate,
    correct_p_values,
    compute_pairwise_distances,
    compute_spectral_correlation,
    compute_spectral_jsd
)
from functions.plots import (
    plot_plv_matrix, 
    plot_plv_map, 
    plot_spectra,
    plot_plv_vs_jsd
)
from functions.io import Recording, save_open_figs

## 2. Analysis Parameters

Configure the analysis before loading any data.

| Parameter | Description | Default |
|---|---|---|
| `keep_cells` | Cell quality labels to include | `("accepted")` |
| `lowpass_cutoff` | Lower bound of bandpass filter (Hz) | `0.01` |
| `highpass_cutoff` | Upper bound of bandpass filter (Hz) | `1.0` |
| `permutation_number` | Number of permutations for statistical analysis | `1000` |

> **Tip:** Set `keep_cells = None` to include all cells regardless of their status label.

In [ ]:
# Cell status labels to retain in the analysis.
# Options typically found in the CSV: 'accepted', 'undecided', 'rejected'
keep_cells = ("undecided")

# Bandpass filter cutoff frequencies (Hz).
# These bracket the TIDA oscillation frequency (~0.05–0.1 Hz).
lowpass_cutoff  = 0.01   # removes very slow baseline drift
highpass_cutoff = 1.0    # removes fast noise above the frequency of interest

# Define the number of permutations for permutation test
permutation_number = 1000

## 3. Load Recording

Select the traces CSV via a file dialog. The `Recording` class automatically:
- Parses the CSV and filters cells by status
- Locates and loads the `*-props.csv` properties file (if present)
- Locates and loads the map image PNG (if present)

**Alternative (manual path):** uncomment the two lines at the bottom of the cell.

In [ ]:
# Open a file dialog to select the recording CSV.
# Props and map image are located automatically from the same folder.
recording = Recording.from_dialog(keep_status=keep_cells)

# --- Manual alternative ---
# recording_path = r"path/to/my_recording.csv"
# recording = Recording(recording_path, keep_status=keep_cells).load()

## 4. Select Analysis Interval

Restrict the analysis to a specific time window. This is useful to:
- Exclude artefacts or unstable recording periods
- Focus on a particular experimental epoch

**Available selection methods:**

| Method | How it works |
|---|---|
| `'single_click'` | Click once to set start; duration set by `window` (seconds) |
| `'double_click'` | Click twice to define start and end |
| `'manual'` | Pass `start_time` and `end_time` directly — no interaction |
| `'block'` | Click within a block defined by detected time-axis jumps |

> **Requires** an interactive matplotlib backend — the cell above sets `%matplotlib qt`.

In [ ]:
# Switch to interactive Qt backend so ginput() works in Jupyter
%matplotlib qt

# Click once on the trace plot to define the interval start.
# The interval will extend for `window` seconds from that point.
recording.select_interval(method='single_click', window=300)

## 5. Bandpass Filtering

A zero-phase Butterworth bandpass filter is applied to each trace to:
- Remove slow baseline drift (below `lowpass_cutoff`)
- Remove high-frequency noise (above `highpass_cutoff`)

The filtered signal is the input to the Hilbert transform for phase extraction.

In [ ]:
# Unpack the selected interval from the recording object
data          = recording.data
time          = recording.time
sampling_rate = recording.sampling_rate

# Apply the bandpass filter to every column (cell) in the DataFrame.
# Uses scipy.signal.filtfilt (zero-phase, forward-backward filtering).
filtered = data.apply(lambda x: bandpass(x, sampling_rate, lowpass_cutoff, highpass_cutoff))

print(f"Filtered {filtered.shape[1]} cells over {filtered.shape[0]} timepoints")
print(f"Sampling rate: {sampling_rate:.4f} Hz | Duration: {time[-1]:.1f} s")

## 6. Phase Extraction and PLV Computation

### 6.1 Instantaneous Phase
The instantaneous phase is extracted via the **Hilbert transform**. 

### 6.2 PLV Matrix
All pairwise PLV values and mean phase differences are computed in a single vectorised pass using `np.einsum`.

In [ ]:
# Extract instantaneous phase for each cell using the Hilbert transform.
# Input must be narrowband (bandpass filtered) for reliable phase estimates.
filtered_phase = filtered.apply(get_phase)

# Compute the full pairwise PLV matrix and mean phase difference matrix.
# plv_einsum expects shape (n_cells, n_timepoints) — transpose accordingly.
PLV, phase_diff = plv_einsum(filtered_phase.T.to_numpy())

# Wrap results in DataFrames for labelled access and easier downstream handling
PLV_df        = pd.DataFrame(PLV,        index=recording.cell_names, columns=recording.cell_names)
phase_diff_df = pd.DataFrame(phase_diff, index=recording.cell_names, columns=recording.cell_names)

print(f"PLV matrix computed: {PLV_df.shape[0]} x {PLV_df.shape[1]} cells")
print(f"Mean PLV (off-diagonal): {PLV_df.values[~np.eye(PLV_df.shape[0], dtype=bool)].mean():.3f}")

## 7. Hierarchical Clustering

The PLV matrix is reordered using **Ward hierarchical clustering** on the PLV-derived distance matrix (`distance = 1 - PLV`). This groups cells with similar phase-locking profiles together, making spatial and functional clusters visually apparent in the heatmap.

In [ ]:
# Obtaining the phase for each trace and calculating the PLV matrix
filtered_phase = filtered.apply(get_phase)

In [ ]:
# Calculate PLV matrix using the optimized einsum function
PLV, phase_diff = plv_einsum(filtered_phase.T.to_numpy()) #Transpose to have shape (n_traces, n_timepoints) for plv_einsum
PLV_df = pd.DataFrame(PLV, index=recording.cell_names, columns=recording.cell_names) #convert to DataFrame for better visualization and handling of labels
phase_diff_df = pd.DataFrame(phase_diff, index=recording.cell_names, columns=recording.cell_names) 

In [ ]:
# Convert similarity (PLV) to distance
dist = 1-PLV_df
# Hierarchical clustering (Ward works well)
Z = linkage(squareform(dist), method='ward')
# Get order of leaves
idx = leaves_list(Z)

PLV_sorted = PLV_df.iloc[idx, idx]  #sort the PLV matrix according to the hierarchical clustering order
phase_diff_sorted = phase_diff_df.iloc[idx,idx]

In [ ]:
# Estimate the peak FFT frequency for each filtered trace
dominant_freqs, frqeq_axis, powers = compute_spectra(filtered.iloc[:,idx], sampling_rate, poly_order=4, hann=True)

slowest_dominant_freq = dominant_freqs.min()
slowest_index         = dominant_freqs.idxmin()

print(f"Dominant frequencies (Hz):\n{dominant_freqs.describe().round(4)}")
print(f"\nSlowest cell: {slowest_index} @ {slowest_dominant_freq:.4f} Hz "
      f"(period ≈ {1/slowest_dominant_freq:.1f} s)")

# min_interval = at least one full oscillation cycle of the slowest cell
# max_interval = full trace length minus one cycle
min_interval = int((1 / slowest_dominant_freq) * sampling_rate)
max_interval = int(filtered_phase.shape[0] - min_interval)
print(f"Surrogate shift range: {min_interval} - {max_interval} samples")

## 9. Permutation (Surrogate) Test

Statistical significance of each PLV is assessed against a null distribution built from surrogate data.

| Surrogate method | Preserves | Best for |
|---|---|---|
| `circular_shift_surrogate` | Autocorrelation, power spectrum | Phase-diff test |
| `phase_randomisation_surrogate` | Power spectrum only | PLV test (stronger null) |

This notebook uses **`phase_randomisation_surrogate`**.

In [ ]:
# Reorder filtered signals to match the clustered PLV matrix
filtered_sorted = filtered.iloc[:, idx]

# Run permutation test — _ discards surrogate arrays and phase-diff p-values
_, p_vals, _, _ = permutation_test(
    filtered_sorted.T.to_numpy(),
    PLV_sorted.to_numpy(),
    phase_diff_sorted.to_numpy(),
    n_permutations=permutation_number,
    plot_surrogates=0,                      # set > 0 to visually inspect surrogates
    surrogate_fn=phase_randomisation_surrogate,
)

upper_tri = p_vals[np.triu_indices(p_vals.shape[0], k=1)]  # k=1 excludes diagonal

print(f"Permutation test complete ({permutation_number} permutations)")
print(f"Raw p-value range: {upper_tri.min():.4f} - {upper_tri.max():.4f}")

## 10. FDR Correction (PLV)

Benjamini-Hochberg FDR correction is applied to control the false discovery rate across all cell pairs.

In [ ]:
# BH FDR correction — returns matrix (for plotting) and flat vector (for export)
p_val_matrix, p_vals_vector = correct_p_values(
    p_vals,
    FDR_correction=True,
    cell_labels=filtered_sorted.columns.values,
    method='bh',
)

# Retain uncorrected p-values for the output CSV
_, p_vals_uncorrected = correct_p_values(
    p_vals, FDR_correction=False, cell_labels=filtered_sorted.columns.values
)
p_vals_uncorrected = p_vals_uncorrected.rename(columns={'p_value': 'p_value_uncorrected'})

n_sig   = (p_vals_vector['p_value'] < 0.05).sum()
n_total = len(p_vals_vector)
print(f"Significant pairs (BH-corrected, α=0.05): {n_sig} / {n_total}")

## 11. Linearise PLV Matrices

Upper-triangle values are extracted from the PLV and phase difference matrices as flat DataFrames, ready to be joined with spectral results in Part 2.

In [ ]:
# Linearise PLV values (upper triangle only)
_, PLV_vals_vector = correct_p_values(
    PLV_sorted.to_numpy(), FDR_correction=False, cell_labels=filtered_sorted.columns.values
)
PLV_vals_vector = PLV_vals_vector.rename(columns={'p_value': 'PLV Value'})

# Linearise phase differences
_, phase_diff_vector = correct_p_values(
    phase_diff_sorted.to_numpy(), FDR_correction=False, cell_labels=filtered_sorted.columns.values
)
phase_diff_vector = phase_diff_vector.rename(columns={'p_value': 'Phase Diff (radians)'})


## 12. Visualisation (Part 1)

### 12.1 PLV Heatmap

In [ ]:
# PLV matrix sorted by Ward clustering; significant pairs annotated with stars
plot_plv_matrix(
    PLV_sorted, p_val_matrix,
    square='True', only_significant=True,
    cmap='viridis', cbar_kws={'label': 'PLV'},
)

### 12.2 Spatial PLV Map

In [ ]:
# Lines between significantly phase-locked pairs, coloured by PLV strength
plot_plv_map(
    recording.props, PLV_vals_vector,
    map=recording.map_img, p_val_vector=p_vals_vector,
)

## 13. Pairwise Distance Computation

In [ ]:
# Euclidean centroid distances — multiply by pixel size (µm/px) to convert units
distance_vector = compute_pairwise_distances(recording.props, PLV_vals_vector.index)

print(f"Distance range: {distance_vector['distance_px'].min():.1f} – "
      f"{distance_vector['distance_px'].max():.1f} px")

---
# PART 2 — Power Spectral Similarity (Jensen-Shannon Distance)
---

## What is JS Distance?

While PLV captures **phase synchrony**, spectral similarity asks a different question:
do two cells oscillate with the same spectral profile, regardless of their phase relationship?

We use the **Jensen-Shannon Distance (JSD)** — the square root of the Jensen-Shannon
Divergence between normalised power spectra.

Each spectrum is first normalised to a probability distribution over frequencies,
so only spectral *shape* is compared.

Unlike a plain Pearson correlation (which is implemented in `functions.utils.compute_spectral_correlation`) on raw PSDs, the normalisation step minimizes the
impact of the shared noise floor that all bandpass-filtered cells have in common, so the distance
reflects genuine differences in oscillatory profile rather than methodological artefact.

Comparing JS Distance with PLV per pair reveals four functional regimes:

| PLV | JS Distance | Interpretation |
|---|---|---|
| High | Low | Shared frequency content **and** consistent phase — strong coupling |
| High | High | Phase-locked despite different spectral profiles |
| Low | Low | Similar frequency content but no consistent phase relationship |
| Low | High | Uncoupled in both senses |

> **Cross-matrix consistency:** `psds` rows are already in the PLV clustering order
> (inherited from `filtered_sorted` in Section 7). Do **not** re-cluster here.

## 14. JS Distance Matrix

`psds` and `freq_axis` are available directly from Section 8 (`compute_spectra`) —
no separate spectrum computation is needed.

`compute_spectral_jsd` normalises each spectrum internally to a probability
distribution, computes pairwise $\sqrt{JSD}$, and returns a symmetric distance
matrix (diagonal = 0, values in [0, 1]).

In [ ]:
# compute_spectral_jsd normalises psds internally — pass raw power spectra.
# psds rows are already in PLV clustering order (from filtered_sorted, Section 7).
jsd_matrix, _ = compute_spectral_jsd(
    powers,
    cell_labels=filtered_sorted.columns,
)

# Summary statistics on upper triangle only (k=1 excludes diagonal)
upper = jsd_matrix.values[np.triu_indices(jsd_matrix.shape[0], k=1)]
print(f"JS Distance matrix: {jsd_matrix.shape[0]} x {jsd_matrix.shape[1]}")
print(f"Mean JS Distance (off-diagonal): {upper.mean():.3f}")
print(f"Range: {upper.min():.3f} – {upper.max():.3f}")

# Linearise JS Distance values (upper triangle only) for export and scatter.
# correct_p_values is reused here purely for its upper-triangle extraction logic.
_, jsd_vector = correct_p_values(
    jsd_matrix.to_numpy(),
    FDR_correction=False,
    cell_labels=filtered_sorted.columns.values,
)
jsd_vector = jsd_vector.rename(columns={'p_value': 'JS Spectral Distance'})

## 15. Visualisation

### 15.1 Overview: PSD per Cell + Dominant Frequency Distribution

A 2-panel figure giving an overview of the spectral landscape of the population:
- **Left:** power spectrum of every cell overlaid, with the median dominant frequency marked
- **Right:** histogram of dominant frequencies across the population

In [ ]:
plot_spectra(frqeq_axis, powers, dominant_freqs, highpass_cutoff)

# Coefficient of variation as a scalar summary of spectral heterogeneity across cells
print(f"Dominant frequency CV: {dominant_freqs.std() / dominant_freqs.mean():.3f}")
print(f"  (CV ≈ 0 = spectrally homogeneous population, CV >> 0 = heterogeneous)")

### 15.2 JS Distance Heatmap

Low values (dark) indicate cells with similar spectral shapes; high values (bright)
indicate spectrally dissimilar pairs. Cell ordering matches the PLV heatmap
(Section 12.1), but the two matrices are not expected to look the same — they
measure fundamentally different properties of the signal.

> JS Distance has no associated parametric p-value, so no significance stars are
> overlaid. Use the scatter plot in Section 15.3 to relate JSD to PLV significance.

In [ ]:
# JS Distance heatmap — low = spectrally similar, high = spectrally dissimilar.
# Cell order matches PLV_sorted for structural comparison, not visual similarity.
plot_plv_matrix(
    jsd_matrix,
    square='True',
    only_significant=False,
    cmap='viridis',
    vmin=0, vmax=1,
    title='JS Distance Matrix (√JSD)',
    cbar_kws={'label': '√JSD'},
)

### 15.3 PLV vs. JS Distance Scatter

Each point is a unique cell pair, coloured by whether the PLV is significant
(BH p < 0.05). Significantly phase-locked pairs with low JS Distance are the
most strongly coupled — consistent phase relationship *and* similar spectral profiles.

Note that the x-axis is now a **distance** (lower = more similar), opposite to the
PLV axis (higher = more synchronised).

In [ ]:
# Colour pairs by PLV significance — red = significant, blue = not significant.
# Index alignment is guaranteed because both vectors were extracted from matrices
# sorted by the same idx (Section 7).

plot_plv_vs_jsd

print(f"PLV-JS Distance correlation: r = {r_overall:.3f}, p = {p_overall:.3g}")
print("(negative r = phase-locked pairs tend to have more similar spectra)")

## 16. Export Combined Results

All per-pair metrics are joined into a single DataFrame and exported. Every row is a
unique cell pair; the index is consistent across all columns because all matrices were
sorted by the same `idx` from Section 7.

| Column | Description |
|---|---|
| `PLV Value` | Phase Locking Value |
| `p_value` | BH-corrected p-value for PLV |
| `p_value_uncorrected` | Uncorrected p-value for PLV |
| `Phase Diff (radians)` | Mean phase difference |
| `distance_px` | Euclidean centroid distance (pixels) |
| `JS Distance` | $\sqrt{JSD}$ between normalised power spectra (Endres & Schindelin, 2003) |

In [ ]:
# Assemble combined results table — pd.concat aligns on the shared pair-label index.
results_df = pd.concat(
    [
        PLV_vals_vector,     # PLV Value
        p_vals_vector,       # p_value (BH-corrected PLV)
        p_vals_uncorrected,  # p_value_uncorrected
        phase_diff_vector,   # Phase Diff (radians)
        distance_vector,     # distance_px
        jsd_vector,          # JS Distance (sqrt JSD)
    ],
    axis=1,
)

save_folder = recording.path.parent

# Save combined CSV alongside the input recording
results_df.to_csv(
    save_folder / 'results_PLV_spectral.csv',
    index=True,
    index_label='Cell(i) vs. Cell(j)',
)
print(f"Results saved to: {save_folder / 'results_PLV_spectral.csv'}")

# Save all open figures as a multi-page PDF
save_open_figs(save_folder=save_folder, save_name='Results_PLV_spectral_figures.pdf')
print(f"Figures saved to: {save_folder / 'Results_PLV_spectral_figures.pdf'}")

results_df